# TradeOps Blue Ocean Discovery Analysis
## Methodology: How we identify undiscovered Ghana trade opportunities

This notebook walks through the scoring framework, data sources, and reasoning behind each 'blue ocean' discovery.

**What is a blue ocean opportunity?**  
A commodity or product where:
1. Ghana has a genuine competitive advantage (climate, geography, raw material access)
2. Global demand is growing or forming
3. No established exporter or importer has yet built a brand or system around it
4. Entry is feasible within GHC 100,000 budget

Reference: Kim & Mauborgne (2005). *Blue Ocean Strategy*. HBR Press.

In [ ]:
import sys
from pathlib import Path

# Add scripts directory to path
scripts_dir = Path('..') / 'scripts'
sys.path.insert(0, str(scripts_dir))

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from discover_opportunities import get_all_discoveries, get_discovery_summary, DISCOVERIES

print('Blue Ocean Discovery module loaded.')
print(f'Total discoveries in database: {len(DISCOVERIES)}')

---
## Step 1: The Scoring Methodology

Each discovery is scored on 5 factors totalling 100 points:

In [ ]:
methodology = {
    'Factor': [
        'Market Saturation (inverted)',
        'Ghana Competitive Advantage',
        'Global Demand Trajectory',
        'Entry Barrier / Budget Fit',
        'First-Mover Advantage'
    ],
    'Weight': ['30%', '25%', '25%', '10%', '10%'],
    'Max Points': [30, 25, 25, 10, 10],
    'How Scored': [
        'Inverted: 0 = many established exporters, 30 = zero commercial exporters worldwide',
        'Access to raw material in Ghana, climate suitability, local processing capacity',
        'CAGR of relevant global market, keyword trend growth, consumer interest signals',
        'Can GHC 100k start it? Low barrier = high score',
        'How long until competitors can replicate? Unique IP/story = higher score'
    ]
}

mdf = pd.DataFrame(methodology)
display(mdf.style.set_properties(**{'text-align': 'left'}))

---
## Step 2: Data Sources Used for Discovery Research

Unlike established commodities (which use UN Comtrade and World Bank APIs), blue ocean discoveries require different signals — the very nature of 'undiscovered' means trade APIs show nothing yet.

In [ ]:
sources = [
    {'Signal Type': 'Scientific literature', 'Source': 'PubMed', 'What we looked for': 'Validated health/nutritional claims for Ghana botanicals'},
    {'Signal Type': 'Market size data', 'Source': 'Grand View Research / Mordor Intelligence', 'What we looked for': 'CAGR figures for relevant product categories'},
    {'Signal Type': 'Consumer trend', 'Source': 'Google Trends (5-year)', 'What we looked for': 'Search volume growth for product keywords'},
    {'Signal Type': 'Trade flows', 'Source': 'ITC Trade Map', 'What we looked for': 'Export values near zero = undiscovered signal'},
    {'Signal Type': 'Retail evidence', 'Source': 'Amazon bestseller lists, Etsy search', 'What we looked for': 'Existing products, prices, review counts, gaps'},
    {'Signal Type': 'Academic research', 'Source': 'CSIR Ghana, FORIG publications', 'What we looked for': 'Documented wild/cultivated resources in Ghana'},
    {'Signal Type': 'Industry reports', 'Source': 'Specialty food trade press, Mintel', 'What we looked for': 'Emerging ingredient trends not yet mainstreamed'},
    {'Signal Type': 'Competitive scan', 'Source': 'LinkedIn, Alibaba.com', 'What we looked for': 'How many companies are already exporting each product'},
]

sdf = pd.DataFrame(sources)
display(sdf)

---
## Step 3: All Discoveries — Full Scored Database

In [ ]:
all_d = get_all_discoveries()

rows = []
for d in all_d:
    rows.append({
        'Name': d['name'].split('(')[0].strip()[:35],
        'Tier': d['blue_ocean_tier'],
        'Direction': d['direction'].upper(),
        'Score /100': d['total_score'],
        'Saturation /30': d['market_saturation_score'],
        'GH Advantage /25': d['ghana_advantage_score'],
        'Demand /25': d['demand_trajectory_score'],
        'Entry /10': d['entry_barrier_score'],
        'First Mover /10': d['first_mover_score'],
        'Margin %': d['gross_margin_pct'],
        'Start GHC': d['starting_order_ghc'],
    })

df = pd.DataFrame(rows)

def colour_tier(val):
    if val == 'EXTREME': return 'background-color: #8b0000; color: white'
    if val == 'HIGH': return 'background-color: #8b4500; color: white'
    if val == 'MEDIUM-HIGH': return 'background-color: #7a6500; color: white'
    return 'background-color: #1a5c1a; color: white'

def colour_score(val):
    if val >= 85: return 'background-color: #1a5c1a; color: white'
    if val >= 70: return 'background-color: #3d7a00; color: white'
    return 'background-color: #b8860b; color: white'

styled = (df.style
    .map(colour_tier, subset=['Tier'])
    .map(colour_score, subset=['Score /100'])
    .format({'Start GHC': 'GHC {:,.0f}', 'Margin %': '{:.0f}%'})
    .set_caption('Blue Ocean Discoveries — Ranked by Total Score')
)
display(styled)

---
## Step 4: Discovery Map — Score vs Gross Margin

In [ ]:
tier_colors = {'EXTREME': '#e74c3c', 'HIGH': '#e67e22', 'MEDIUM-HIGH': '#f1c40f', 'MEDIUM': '#27ae60'}

fig = go.Figure()
for tier, color in tier_colors.items():
    sub = df[df['Tier'] == tier]
    if sub.empty: continue
    fig.add_trace(go.Scatter(
        x=sub['Score /100'], y=sub['Margin %'],
        mode='markers+text',
        name=tier,
        marker=dict(size=16, color=color),
        text=sub['Name'],
        textposition='top center',
    ))

fig.add_hline(y=20, line_dash='dash', line_color='gray', annotation_text='20% margin minimum')
fig.add_vline(x=70, line_dash='dash', line_color='gray', annotation_text='Score threshold 70')
fig.update_layout(
    title='Blue Ocean Discovery Map: Score vs Margin (ideal = top-right quadrant)',
    xaxis_title='Blue Ocean Score /100',
    yaxis_title='Gross Margin %',
    height=550,
    template='plotly_dark',
)
fig.show()

---
## Step 5: Factor Radar — Top 4 Discoveries

In [ ]:
top4 = all_d[:4]
categories = ['Saturation (inv)', 'GH Advantage', 'Demand', 'Entry Barrier', 'First Mover']
max_vals = [30, 25, 25, 10, 10]
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#27ae60']

fig = go.Figure()
for i, d in enumerate(top4):
    scores = [
        d['market_saturation_score'],
        d['ghana_advantage_score'],
        d['demand_trajectory_score'],
        d['entry_barrier_score'],
        d['first_mover_score'],
    ]
    pct = [s / m * 100 for s, m in zip(scores, max_vals)]
    fig.add_trace(go.Scatterpolar(
        r=pct + [pct[0]],
        theta=categories + [categories[0]],
        fill='toself',
        name=d['name'].split('(')[0].strip()[:25],
        line=dict(color=colors[i]),
        fillcolor=colors[i].replace(')', ', 0.15)').replace('rgb', 'rgba'),
        opacity=0.85,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 100], visible=True, tickformat='.0f')),
    title='Top 4 Discoveries — Factor Breakdown (% of max per factor)',
    height=550,
    template='plotly_dark',
)
fig.show()

---
## Step 6: Value / Price Trend Analysis

How has the estimated export value of each discovery moved over time? (Based on published research and market data 2021-2025)

In [ ]:
fig = go.Figure()
colors_trend = px.colors.qualitative.Set2

for i, d in enumerate(all_d):
    pt = d.get('price_trend', {})
    if not pt: continue
    years = list(pt.keys())
    prices = list(pt.values())
    name = d['name'].split('(')[0].strip()[:20]
    fig.add_trace(go.Scatter(
        x=years, y=prices,
        mode='lines+markers',
        name=name,
        line=dict(color=colors_trend[i % len(colors_trend)]),
        marker=dict(size=6),
        hovertemplate=f'{name}<br>%{{x}}: $%{{y}}/kg<extra></extra>',
    ))

fig.update_layout(
    title='Estimated Export Value Trends (USD/kg) — All Discoveries 2021-2025',
    xaxis_title='Year',
    yaxis_title='USD per kg (estimated)',
    height=500,
    template='plotly_dark',
    legend=dict(orientation='v', x=1.02),
)
fig.show()
print('Note: 2025 values are estimates based on trend extrapolation and market research.')

---
## Step 7: Discoveries vs Established Commodities

Why pursue a discovery if established opportunities exist? The key insight: **discoveries have lower competition and higher margin, but require more market education.**

In [ ]:
established = [
    {'Commodity': 'Shea butter', 'Type': 'Established', 'Score': 78, 'Margin %': 30,
     'Competition': 'High', 'Time to First Sale (days)': 30, 'Education Required': 'Low'},
    {'Commodity': 'Cashew nuts', 'Type': 'Established', 'Score': 71, 'Margin %': 22,
     'Competition': 'High', 'Time to First Sale (days)': 45, 'Education Required': 'Low'},
    {'Commodity': 'Moringa powder', 'Type': 'Established', 'Score': 75, 'Margin %': 45,
     'Competition': 'Medium', 'Time to First Sale (days)': 60, 'Education Required': 'Medium'},
    {'Commodity': 'Rice (import)', 'Type': 'Established', 'Score': 68, 'Margin %': 18,
     'Competition': 'Very High', 'Time to First Sale (days)': 30, 'Education Required': 'None'},
]

discoveries_comp = [
    {'Commodity': d['name'].split('(')[0].strip()[:25], 'Type': 'Discovery',
     'Score': d['total_score'], 'Margin %': d['gross_margin_pct'],
     'Competition': 'None/Zero',
     'Time to First Sale (days)': 14,  # DTC e-commerce is fast
     'Education Required': 'High'}
    for d in all_d[:4]
]

comp_df = pd.DataFrame(established + discoveries_comp)

def colour_type(val):
    if val == 'Discovery': return 'background-color: #8b0000; color: white'
    return 'background-color: #1a3c5c; color: white'

display(comp_df.style
    .map(colour_type, subset=['Type'])
    .set_caption('Discoveries vs Established: Key Differences')
    .format({'Margin %': '{:.0f}%', 'Score': '{:.0f}'})
)

In [ ]:
fig = px.scatter(
    comp_df,
    x='Score', y='Margin %',
    color='Type',
    text='Commodity',
    size=[14] * len(comp_df),
    color_discrete_map={'Established': '#3498db', 'Discovery': '#e74c3c'},
    title='Discoveries vs Established: Score vs Margin',
    labels={'Score': 'Opportunity Score /100', 'Margin %': 'Gross Margin %'},
    template='plotly_dark',
)
fig.update_traces(textposition='top center')
fig.add_hline(y=20, line_dash='dash', line_color='gray', annotation_text='Min 20% margin')
fig.update_layout(height=500)
fig.show()

---
## Step 8: Budget Analysis — Which Discoveries Fit GHC 100,000?

Applying the Weiss constraint: max single order = GHC 60,000 (leaving operating capital).

In [ ]:
MAX_ORDER = 60_000
BUDGET = 100_000

budget_rows = []
for d in all_d:
    start = d['starting_order_ghc']
    brand_est = 6_000  # estimated branding/packaging budget
    certs_est = 2_500  # estimated first certification
    shipping_est = 3_000  # estimated initial shipping
    total_est = start + brand_est + certs_est + shipping_est
    feasible = total_est <= BUDGET
    budget_rows.append({
        'Discovery': d['name'].split('(')[0].strip()[:30],
        'Stock Cost (GHC)': start,
        'Brand + Pack (GHC)': brand_est,
        'First Cert (GHC)': certs_est,
        'Shipping Est. (GHC)': shipping_est,
        'Total Required (GHC)': total_est,
        'Within Budget?': 'YES' if feasible else 'STRETCH',
        'Margin %': d['gross_margin_pct'],
    })

bdf = pd.DataFrame(budget_rows)

def colour_feasible(val):
    if val == 'YES': return 'background-color: #1a5c1a; color: white'
    return 'background-color: #8b0000; color: white'

display(bdf.style
    .map(colour_feasible, subset=['Within Budget?'])
    .format({c: 'GHC {:,.0f}' for c in bdf.columns if 'GHC' in c})
    .format({'Margin %': '{:.0f}%'})
    .set_caption(f'Budget Analysis — GHC {BUDGET:,} total, max order GHC {MAX_ORDER:,}')
)

feasible_count = sum(1 for r in budget_rows if r['Within Budget?'] == 'YES')
print(f'\n{feasible_count}/{len(all_d)} discoveries fit within GHC {BUDGET:,} budget.')

---
## Step 9: First-Mover Timeline Simulation

Modelling when competitors are likely to enter each discovery market, and what the first-mover window looks like.

In [ ]:
import numpy as np

# Estimate: months until competitor enters (based on first-mover score)
# Higher first-mover score = longer window
for d in all_d:
    d['competitive_window_months'] = round(d['first_mover_score'] * 2.5 + d['market_saturation_score'] * 0.5)

months = list(range(1, 25))

fig = go.Figure()
colors_fm = px.colors.qualitative.Bold

for i, d in enumerate(all_d[:5]):
    name = d['name'].split('(')[0].strip()[:20]
    window = d['competitive_window_months']
    # Revenue curve: grows during window, slows after competition enters
    revenue = []
    for m in months:
        if m <= window:
            # First-mover growth phase
            revenue.append(round(10_000 * (1 + 0.15) ** m))
        else:
            # Competition enters, growth slows but doesn't stop (brand advantage)
            base = revenue[-1] if revenue else 10_000
            revenue.append(round(base * (1 + 0.05)))

    fig.add_trace(go.Scatter(
        x=months, y=revenue,
        mode='lines',
        name=f'{name} (window: {window}mo)',
        line=dict(color=colors_fm[i % len(colors_fm)], width=2),
        hovertemplate=f'{name}<br>Month %{{x}}: GHC %{{y:,.0f}}<extra></extra>',
    ))
    # Mark when competition enters
    if window <= 24:
        fig.add_vline(
            x=window,
            line_dash='dot',
            line_color=colors_fm[i % len(colors_fm)],
            opacity=0.5,
        )

fig.update_layout(
    title='Simulated Revenue Growth: Top 5 Discoveries (competition entry marked with dotted line)',
    xaxis_title='Month from start',
    yaxis_title='Estimated Monthly Revenue (GHC)',
    height=500,
    template='plotly_dark',
    legend=dict(orientation='v', x=1.01),
)
fig.show()
print('Dotted lines = estimated month when competitors begin entering the market.')
print('After that point, brand loyalty and supplier lock-in become critical moats.')

---
## Step 10: Top Pick Analysis — Prekese Deep Dive

Detailed reasoning for the #1 ranked discovery.

In [ ]:
top = all_d[0]
print(f'Top discovery: {top["name"]}')
print(f'Blue Ocean Tier: {top["blue_ocean_tier"]}')
print(f'Total Score: {top["total_score"]}/100')
print()
print('Why this is #1:')
print(top['why_undiscovered'])
print()
print(f'Ghana producer price: ${top.get("ghana_producer_price_usd_kg", "N/A")}/kg')
print(f'Target export price:  ${top.get("estimated_export_price_usd_kg", "N/A")}/kg')
print(f'Estimated margin:     {top["gross_margin_pct"]}%')
print(f'Starting order size:  GHC {top["starting_order_ghc"]:,}')
print()
print('First-mover actions:')
for i, action in enumerate(top.get('first_mover_actions', []), 1):
    print(f'  {i}. {action}')
print()
print('Target buyers:')
for b in top.get('target_buyers', []):
    print(f'  - {b["name"]} ({b["type"]})')
    print(f'    Approach: {b["approach"][:100]}...')

In [ ]:
# Financial projection for top pick
USD_GHC = 15.4
top_item = all_d[0]

source_price_ghc_kg = (top_item.get('ghana_producer_price_usd_kg', 2) or 2) * USD_GHC
export_price_ghc_kg = (top_item.get('estimated_export_price_usd_kg', 8) or 8) * USD_GHC
start_qty = top_item['starting_qty_kg']
start_order = top_item['starting_order_ghc']

packaging_cost = 3_500
freight_cost = 2_500
cert_cost = 1_500
total_investment = start_order + packaging_cost + freight_cost + cert_cost
gross_revenue = start_qty * export_price_ghc_kg
gross_margin_ghc = gross_revenue - total_investment
margin_pct = gross_margin_ghc / gross_revenue * 100

print('=== First Order Financials ===')
print(f'Product ({start_qty} kg @ GHC {source_price_ghc_kg:.1f}/kg): GHC {start_order:,.0f}')
print(f'Packaging & branding:                                   GHC {packaging_cost:,.0f}')
print(f'Freight (DHL Express to UK):                            GHC {freight_cost:,.0f}')
print(f'Certificates (COA, phytosanitary):                      GHC {cert_cost:,.0f}')
print(f'TOTAL INVESTMENT:                                        GHC {total_investment:,.0f}')
print()
print(f'Revenue ({start_qty} kg @ GHC {export_price_ghc_kg:.1f}/kg):            GHC {gross_revenue:,.0f}')
print(f'GROSS PROFIT:                                            GHC {gross_margin_ghc:,.0f}')
print(f'GROSS MARGIN:                                            {margin_pct:.1f}%')
print()
passed = margin_pct >= 20
print(f'Weiss 20% margin test: {"PASSED" if passed else "FAILED"}')

---
## Conclusion

### Key findings from the Blue Ocean Analysis:

1. **4 EXTREME blue ocean opportunities** exist in Ghana's botanical/food space — with zero established international competition.

2. **Margin potential is significantly higher** in discoveries (50-90%) than in established commodities (15-30%), because:
   - No price competition yet
   - DTC e-commerce allows retail pricing (not wholesale)
   - Premium positioning through storytelling

3. **Entry cost is low** — the highest-scoring discoveries (Prekese, Alligator pepper, Dawadawa) can all be started with GHC 15,000-25,000 product cost, within the GHC 100,000 budget.

4. **The first-mover window is 12-24 months** — after which imitators will enter. The key moats to build are: brand, supplier contracts, certifications, and customer relationships.

5. **Recommended first pick: Prekese** — unique to West Africa, zero global commercial exporters, validated health claims, diaspora demand exists, can start DTC on Etsy within 30 days.

---
*Analysis methodology: Blue Ocean Strategy framework (Kim & Mauborgne, 2005) + Weiss import/export business model constraints.*  
*Data: ITC Trade Map, PubMed, Amazon/Etsy market research, specialty trade publications 2024-2025.*